# Build an Agent Swarm with CrewAI and Gemini 3.5 Flash

A beginner-friendly, end-to-end guide to orchestrating multiple AI agents with **[CrewAI](https://docs.crewai.com/)** powered by **Google's Gemini 3.5 Flash** model.

By the end of this notebook you will have built a small **swarm of four cooperating agents** (Researcher → Writer → Reviewer → Coordinator) that can take a single topic, do research on it, draft a blog post, review it, and hand you a polished final article.

> **A note on the model name.** Google currently advertises its Flash family as **Gemini 3.5 Flash** on the official [DeepMind Gemini Flash page](https://deepmind.google/models/gemini/flash/) and in the official [Gemini API + CrewAI example](https://ai.google.dev/gemini-api/docs/crewai-example). The model string we will use throughout this guide is `gemini/gemini-3.5-flash` — the `gemini/` prefix is the routing prefix that [LiteLLM](https://docs.litellm.ai/docs/providers/gemini) (used under the hood by CrewAI) requires to talk to Google.


## 1. What we are building

We will build a small **content-production swarm** with the following moving parts:

| Component | Role | Responsibility |
| --- | --- | --- |
| **Researcher Agent** | Gathers facts | Searches its knowledge for up-to-date points on the topic |
| **Writer Agent** | Drafts content | Turns the research notes into a blog post draft |
| **Reviewer Agent** | Polishes output | Edits the draft for clarity, tone, and correctness |
| **Manager Agent** | Coordinates | Oversees the work and produces a final summary |
| **Gemini 3.5 Flash** | The LLM brain | Powers all four agents via the Gemini API |
| **CrewAI** | The orchestrator | Wires the agents and tasks together into a single `kickoff()` |

**Example task we will run:** &gt; *"Research and write a short blog post about the benefits of local-first AI agents."*

Everything you write below lives in standard Python — no exotic frameworks, no cloud lock-in beyond the Gemini API itself.


## 2. Architecture overview

Here is what the swarm looks like at runtime. Each box is a CrewAI `Agent`, the arrows are `Task` dependencies, and the dotted arrow shows the optional `manager_llm` delegation path used by `Process.hierarchical`.

```mermaid
flowchart LR
    U[You / User Input] -->|topic| R[Researcher Agent]
    R -->|research notes| W[Writer Agent]
    W -->|draft post| RV[Reviewer Agent]
    RV -->|polished post| M[Manager / Coordinator Agent]
    M -->|final answer| OUT([Final blog post])

    G{{Gemini 3.5 Flash}} -. powers .-> R
    G -. powers .-> W
    G -. powers .-> RV
    G -. powers .-> M
```

**Key idea.** Each agent has its own *role*, *goal*, and *backstory* but they all share the same LLM. CrewAI takes care of formatting prompts, passing outputs from one task to the next, and (optionally) letting a manager agent decide what to run next.

We will use `Process.sequential` for clarity — it runs the tasks in the order we list them. At the end of the notebook, we will show how to switch to `Process.hierarchical` if you want a manager-driven workflow.


## 3. Install dependencies

Run the cell below once. It installs:

- `crewai` and `crewai-tools` — the multi-agent framework.
- `google-genai` — Google's **modern, maintained** Python SDK (the older `google-generativeai` package was deprecated on 30 Nov 2025).
- `litellm` — the routing layer CrewAI uses to call Gemini.
- `python-dotenv` — to load your API key from a `.env` file.
- `ipykernel` — so this notebook can run as a Jupyter kernel.

If you are running this in **Google Colab** the `!pip install` prefix works as-is. If you are running locally, you can also use `pip install -r requirements.txt` from the repo root.


In [1]:
# %pip install --quiet -U \
#     crewai \
#     crewai-tools \
#     google-genai \
#     litellm \
#     python-dotenv \
#     olostep
#     nest-asyncio

# Print installed versions so we can verify the install worked.
import importlib.metadata as md
for pkg in ["crewai", "crewai-tools", "google-genai", "litellm", "python-dotenv"]:

    print(f"{pkg:>16}  {md.version(pkg)}")


          crewai  1.14.6
    crewai-tools  1.14.6
    google-genai  2.8.0
         litellm  1.87.1
   python-dotenv  1.2.2


## 4. Set up API keys securely

**Never hardcode API keys in a notebook.** We will load them from a `.env` file in the project root.

### 4.1 Get a Gemini API key

1. Go to [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey).
2. Click **Create API key** and copy the value.
3. Create a `.env` file next to this notebook (or at the repo root) with the following content:

```dotenv
GEMINI_API_KEY=your-real-key-here
```

> You can also use `GOOGLE_API_KEY` — CrewAI accepts either name. The repo ships a template called `.env.example` you can copy:
>
> ```bash
> cp .env.example .env       # macOS / Linux
> copy .env.example .env     # Windows
> ```


In [2]:
"""Load the API key from .env and sanity-check it."""

import os

from pathlib import Path

from dotenv import load_dotenv



# Look for a .env file in the current working directory and one level up.

for candidate in (Path.cwd() / ".env", Path.cwd().parent / ".env"):

    if candidate.exists():

        load_dotenv(candidate)

        print(f"Loaded environment from: {candidate}")

        break

else:

    print("No .env file found. Falling back to OS-level environment variables.")



# CrewAI / LiteLLM will read either name; we surface both for clarity.

api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

if not api_key or api_key == "your-gemini-api-key-here":

    raise RuntimeError(

        "No Gemini API key found. "

        "Set GEMINI_API_KEY (or GOOGLE_API_KEY) in your .env file or shell environment. "

        "Get a free key at https://aistudio.google.com/app/apikey"

    )



# Mask the key in the printout so we don't leak it in screenshots.

masked = api_key[:4] + "..." + api_key[-4:]

print(f"Gemini API key loaded OK ({masked}, length={len(api_key)}).")


No .env file found. Falling back to OS-level environment variables.
Gemini API key loaded OK (AIza...mWCo, length=39).


## 5. Configure Gemini Flash with CrewAI

CrewAI talks to Gemini through [LiteLLM](https://docs.litellm.ai/docs/providers/gemini), so the model name has the form `gemini/<google-model-id>`. The CrewAI docs recommend `temperature=1.0` for Gemini 3.x family models ([source](https://ai.google.dev/gemini-api/docs/crewai-example)).

We will create **one shared LLM object** and pass it to every agent. This is simpler than giving each agent its own client and is the pattern used in the official Google + CrewAI example.


In [3]:
"""Configure the Gemini 3.5 Flash LLM once and reuse it for every agent."""

from crewai import LLM



# The 'gemini/' prefix tells LiteLLM to route to Google. The suffix

# (gemini-3.5-flash) is the actual Google model id.

GEMINI_MODEL = "gemini/gemini-3.5-flash"



gemini_llm = LLM(

    model=GEMINI_MODEL,

    api_key=os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY"),

    temperature=1.0,  # Google recommends 1.0 for the Gemini 3 family.

)



print(f"LLM ready: {gemini_llm.model}")

print("Provider API: Google Gemini (via LiteLLM)")


LLM ready: gemini-3.5-flash
Provider API: Google Gemini (via LiteLLM)


### 5.1 Give the Researcher live web access with Olostep

Right now the Researcher only knows what Gemini was trained on. To make it work on **fresh, verifiable** information we attach two custom tools built on top of the [Olostep Python SDK](https://docs.olostep.com/sdks/python) — the Web Data API for AI agents:

- **`OlostepSearchTool`** — runs a natural-language web search and, if asked, scrapes each result page in the same round-trip. Uses Olostep's `/v1/searches` endpoint ([reference](https://docs.olostep.com/api-reference/searches/create)).
- **`OlostepScrapeTool`** — scrapes a single URL as clean markdown. Uses Olostep's `/v1/scrapes` endpoint ([reference](https://docs.olostep.com/api-reference/scrapes/create)).

Both tools follow the standard [CrewAI `BaseTool` pattern](https://docs.crewai.com/en/concepts/tools) and read `OLOSTEP_API_KEY` from the environment, so you don't have to pass the key in code.

**Get a free Olostep key** at <https://www.olostep.com/dashboard/> and add it to your `.env` (the notebook's `.env.example` already has the placeholder).


> **Install / upgrade note.** The Olostep SDK is pulled in by the install cell at the top of the notebook. If you ran the notebook before this section was added, run `%pip install -U olostep` once in a new cell before continuing.


In [4]:
"""Custom CrewAI tools backed by the Olostep Python SDK.

    Docs: https://docs.olostep.com/sdks/python"""

from pydantic import Field

from crewai.tools import BaseTool

try:

    from olostep import Olostep, Olostep_BaseError

except ImportError:

    raise ImportError(

        "The 'olostep' package is not installed. Run `%pip install -U olostep`."

    )



olostep_api_key = os.getenv("OLOSTEP_API_KEY")

if not olostep_api_key or olostep_api_key == "your-olostep-api-key-here":

    print("Olostep tools will be created but will fail at call time "

          "unless OLOSTEP_API_KEY is set in your .env.")

else:

    print(f"Olostep API key loaded ({olostep_api_key[:4]}...{olostep_api_key[-4:]}).")



# One shared sync client; Olostep recommends it for scripts and notebooks.

_olostep_client = Olostep(api_key=olostep_api_key) if olostep_api_key else None





class OlostepSearchTool(BaseTool):

    """Search the live web with a natural-language query and optionally

    scrape each returned URL in the same round-trip. Returns a markdown

    block with the title, URL, description, and (when scrape=True) the

    full markdown body of every result."""



    name: str = "olostep_web_search"

    description: str = (

        "Useful for finding up-to-date, real-world information on the web. "

        "Input: a clear natural-language query. Optional flags: limit "

        "(1-25, default 5), scrape (true/false, default true), "

        "include_domains (list of bare hosts, e.g. ['bbc.com'])."

    )



    limit: int = Field(default=5, ge=1, le=25)

    scrape: bool = Field(default=True)



    def _run(self, query: str, limit: int | None = None,

             scrape: bool | None = None,

             include_domains: list[str] | None = None) -> str:

        if _olostep_client is None:

            return "OLOSTEP_API_KEY is not set. Add it to your .env file."

        kwargs = {"query": query, "limit": limit or self.limit}

        if scrape or (scrape is None and self.scrape):

            kwargs["scrape_options"] = {"formats": ["markdown"], "timeout": 25}

        if include_domains:

            kwargs["include_domains"] = include_domains

        try:

            search = _olostep_client.searches.create(**kwargs)

        except Olostep_BaseError as e:

            return f"Olostep search error: {type(e).__name__}: {e}"



        if not search.links:

            return f"No results found for query: {query!r}"



        chunks = [f"## Search results for: {query!r}\n"]

        for i, link in enumerate(search.links, 1):

            url = link.get("url", "")

            title = link.get("title") or url

            desc = link.get("description") or ""

            chunks.append(f"### {i}. {title}\n{url}\n{desc}\n")

            md_body = link.get("markdown_content")

            if md_body:

                # Truncate each page so the agent's context window stays sane.

                body = md_body if len(md_body) <= 4000 else md_body[:4000] + "\n...\n[truncated]"

                chunks.append(body + "\n")

        return "\n".join(chunks)





class OlostepScrapeTool(BaseTool):

    """Scrape a single URL and return its content as clean markdown."""



    name: str = "olostep_scrape_url"

    description: str = (

        "Useful when you have an exact URL (e.g. a research paper, a "

        "company blog post, a Wikipedia page) and need its full text. "

        "Input: a single http(s) URL."

    )



    def _run(self, url: str, max_chars: int = 8000) -> str:

        if _olostep_client is None:

            return "OLOSTEP_API_KEY is not set. Add it to your .env file."

        if not url.startswith(("http://", "https://")):

            return f"Invalid URL: {url!r} (must start with http:// or https://)"

        try:

            result = _olostep_client.scrapes.create(

                url_to_scrape=url, formats=["markdown"]

            )

        except Olostep_BaseError as e:

            return f"Olostep scrape error: {type(e).__name__}: {e}"

        body = result.markdown_content or ""

        if not body:

            return f"No content returned for {url}"

        if len(body) > max_chars:

            body = body[:max_chars] + "\n...\n[truncated]"

        return f"## Content of {url}\n\n{body}"





# Instantiate the tools so they can be attached to agents.

olostep_search_tool = OlostepSearchTool()

olostep_scrape_tool = OlostepScrapeTool()

print(f"Tools ready: {olostep_search_tool.name}, {olostep_scrape_tool.name}")


Olostep API key loaded (olos...JisS).
Tools ready: olostep_web_search, olostep_scrape_url


In [5]:
"""Quick smoke-test of the search tool.

    Skipped automatically if no Olostep key is configured."""

if _olostep_client is None:

    print("Skipping demo: OLOSTEP_API_KEY is not set.")

else:

    out = olostep_search_tool._run(

        query="local-first AI agents benefits",

        limit=3,

        scrape=True,

    )

    preview = out[:600] + ("..." if len(out) > 600 else "")

    print(preview)


## Search results for: 'local-first AI agents benefits'

### 1. Local AI Agents In 26 Minutes - YouTube
https://www.youtube.com/watch?v=M-NTwkM3VwM
Sign up now at https://grammarly.com/tina In this video I explain the fundamentals of local AI agents ... Advantage Of Local AI Agents 25:45 Quiz ...

![Thumbnail (1920x1080)](https://i.ytimg.com/vi/M-NTwkM3VwM/maxresdefault.jpg)

# [Local AI Agents In 26 Minutes](https://www.youtube.com/watch?v=M-NTwkM3VwM)

**Visibility**: Public
**Uploaded by**: [Tina Huang](https://www.youtube.com/channel/UC2UXDak6o7rBm23k3Vv5dww)
**Uploaded at**: 2026-04-15T06...


In [6]:
"""Quick smoke-test of the scrape tool on a known stable URL."""

if _olostep_client is None:

    print("Skipping demo: OLOSTEP_API_KEY is not set.")

else:

    out = olostep_scrape_tool._run("https://en.wikipedia.org/wiki/Local-first_software")

    preview = out[:500] + ("..." if len(out) > 500 else "")

    print(preview)


## Content of https://en.wikipedia.org/wiki/Local-first_software

Local-first software - Wikipedia                     

  [![](/static/images/icons/enwiki-25.svg) ![Wikipedia](/static/images/mobile/copyright/wikipedia-wordmark-en-25.svg) ![The Free Encyclopedia](/static/images/mobile/copyright/wikipedia-tagline-en-25.svg)](/wiki/Main_Page)

[Search](/wiki/Special:Search "Search Wikipedia [f]")

Search

Local-first software

 Add languages

[Add links](https://www.wikidata.o...
/root/venv/lib/python3.13/site-packages/olostep/backend/caller.py:491: UserWarning: The API response from 'Create Scrape' contains 1 extra field(s) not defined in the SDK model: cost_usd. This may indicate new API features. Please check for a newer SDK version on Slack (best place to ask), PyPI or Github. (current: 1.0.4). Visit https://docs.olostep.com/sdks/python for updates.
  return await self._invoke(


> **Heads-up on model names.** If you see a `404 models/not-found` error later, it usually means Google renamed the model. As of this writing the canonical name is `gemini-3.5-flash`; if Google rotates it you can swap in `gemini/gemini-2.5-flash` (a stable, slightly older Flash) without any other code changes. The model name is the **only** thing you need to keep in sync.


## 6. Create the agents

Every CrewAI `Agent` is defined by a `role`, a `goal`, and a `backstory`. Think of the role as their *job title*, the goal as the *outcome they are chasing*, and the backstory as their *résumé* — it primes the LLM to behave like that kind of expert.

We will create four agents:


In [7]:
"""Agent 1 of 4 — the Researcher."""

from crewai import Agent



researcher = Agent(

    role="Senior Research Analyst",

    goal=(

        "Find accurate, up-to-date, and well-sourced information "

        "about {topic}. Produce a short list of concrete points that "

        "a writer could turn into a blog post."

    ),

    backstory=(

        "You are a meticulous research analyst who specialises in "

        "distilling complex technology topics into a handful of clear, "

        "verifiable points. You cite specific concepts, tools, and "

        "real-world examples wherever possible."

    ),

    llm=gemini_llm,
    tools=[olostep_search_tool, olostep_scrape_tool],  # live web via Olostep

    verbose=True,           # show the agent's reasoning while it works

    allow_delegation=False, # keeps the swarm simple and predictable

)



print(f"Created agent: {researcher.role}")


Created agent: Senior Research Analyst


In [8]:
"""Agent 2 of 4 — the Writer."""

writer = Agent(

    role="Blog Post Writer",

    goal=(

        "Using the researcher's notes, write a clear, engaging, and "

        "well-structured blog post of about 400 words on {topic}. "

        "Use a friendly, professional tone and short paragraphs."

    ),

    backstory=(

        "You are a seasoned tech writer who has published hundreds of "

        "posts for developer audiences. You excel at turning dry "

        "research into readable prose with a strong opening, a clear "

        "middle, and a memorable closing."

    ),

    llm=gemini_llm,

    verbose=True,

    allow_delegation=False,

)



print(f"Created agent: {writer.role}")


Created agent: Blog Post Writer


In [9]:
"""Agent 3 of 4 — the Reviewer."""

reviewer = Agent(

    role="Senior Editor",

    goal=(

        "Review the writer's draft for clarity, factual accuracy, and "

        "tone. Return a polished final version of the blog post that "

        "is ready to publish."

    ),

    backstory=(

        "You are a detail-oriented senior editor with 15 years of "

        "experience polishing technology articles. You cut fluff, "

        "fix awkward sentences, and make sure every claim is supported."

    ),

    llm=gemini_llm,

    verbose=True,

    allow_delegation=False,

)



print(f"Created agent: {reviewer.role}")


Created agent: Senior Editor


In [10]:
"""Agent 4 of 4 — the Manager / Coordinator."""

manager = Agent(

    role="Content Coordinator",

    goal=(

        "Look at the research notes, draft, and reviewed post, then "

        "produce a concise final summary for the user that includes "

        "the polished blog post and a short bullet list of the key "

        "takeaways."

    ),

    backstory=(

        "You are a calm, organised editorial lead. You make sure the "

        "final deliverable to the user is tight, well-formatted, and "

        "ready to ship."

    ),

    llm=gemini_llm,

    verbose=True,

    allow_delegation=False,

)



print(f"Created agent: {manager.role}")

print(f"\nTotal agents in the swarm: {len([researcher, writer, reviewer, manager])}")


Created agent: Content Coordinator

Total agents in the swarm: 4


## 7. Define tasks for each agent

A `Task` in CrewAI pairs an agent with a `description` (the prompt) and an `expected_output` (the contract for what the agent should produce). Setting `expected_output` is important — it tells the LLM *exactly* what shape the final answer should take.

We will create one task per agent, in the order they should run.


In [11]:
"""Define one Task per agent. The `context=[...]` list wires the

    output of the previous task into the prompt of the next one."""

from crewai import Task



research_task = Task(

    description=(

        "Research the topic '{topic}'. Produce a bullet list of 5-7 "

        "concrete, verifiable points that would make a strong blog post. "

        "For each point, include a one-sentence rationale."

    ),

    expected_output=(

        "A markdown bullet list of 5-7 points, each with a one-sentence "

        "explanation. No prose before or after the list."

    ),

    agent=researcher,

)



write_task = Task(

    description=(

        "Using the research notes above, write a ~400-word blog post "

        "on '{topic}'. Structure it as: a one-line hook, a 2-3 "

        "sentence intro, 3 short body sections (one per key point), "

        "and a 1-2 sentence conclusion."

    ),

    expected_output=(

        "A complete blog post of roughly 400 words, written in markdown, "

        "with a clear hook, intro, three body sections, and a conclusion."

    ),

    agent=writer,

    context=[research_task],  # gets the research output as extra context

)



review_task = Task(

    description=(

        "Review the blog post draft. Fix any factual, clarity, or tone "

        "issues. Return the polished final version of the post — do not "

        "include a list of edits, only the cleaned-up text."

    ),

    expected_output=(

        "A polished, publish-ready blog post in markdown. No commentary, "

        "no 'changes made' notes — just the final post."

    ),

    agent=reviewer,

    context=[write_task],

)



coordinate_task = Task(

    description=(

        "Look at the full chain (research notes, draft, polished post). "

        "Produce a final answer to the user that contains: "

        "(1) the polished blog post, and "

        "(2) a 3-bullet 'Key Takeaways' summary."

    ),

    expected_output=(

        "Markdown with two sections: '## Blog Post' (the polished post) "

        "and '## Key Takeaways' (exactly 3 bullets)."

    ),

    agent=manager,

    context=[research_task, write_task, review_task],

)



tasks = [research_task, write_task, review_task, coordinate_task]

print(f"Defined {len(tasks)} tasks:")

for t in tasks:

    print(f"  - {t.agent.role:<26}  ->  {t.expected_output[:60]}...")


Defined 4 tasks:
  - Senior Research Analyst     ->  A markdown bullet list of 5-7 points, each with a one-senten...
  - Blog Post Writer            ->  A complete blog post of roughly 400 words, written in markdo...
  - Senior Editor               ->  A polished, publish-ready blog post in markdown. No commenta...
  - Content Coordinator         ->  Markdown with two sections: '## Blog Post' (the polished pos...


## 8. Build the CrewAI workflow

A `Crew` is the runtime that takes a list of agents + tasks and a `Process` type, and orchestrates them. We use `Process.sequential` for clarity — tasks run in the order we list them, and each task's output becomes context for the next one (we already wired that up via `context=[...]` above).

If you want a *true* manager-driven workflow, switch `process=Process.hierarchical` and pass `manager_llm=gemini_llm`. We show both options below.


In [12]:
"""Assemble the agents + tasks into a Crew."""

from crewai import Crew, Process



crew = Crew(

    agents=[researcher, writer, reviewer, manager],

    tasks=tasks,

    process=Process.sequential,   # tasks run in the order listed above

    verbose=True,                 # show step-by-step agent logs

    memory=False,                 # flip to True to enable shared short-term memory

)



print("Crew assembled.")

print(f"  Agents : {[a.role for a in crew.agents]}")

print(f"  Tasks  : {len(crew.tasks)}")

print(f"  Process: {crew.process}")


Crew assembled.
  Agents : ['Senior Research Analyst', 'Blog Post Writer', 'Senior Editor', 'Content Coordinator']
  Tasks  : 4
  Process: Process.sequential


## 9. Run the agent swarm on an example topic

Time to actually run the swarm! The `kickoff()` method starts the workflow. We pass the topic as an `inputs` dict — any `{topic}` placeholder in the task descriptions is filled in automatically.

**Example topic:** &gt; *"The benefits of local-first AI agents."*

Because `verbose=True` is set, you will see each agent's reasoning in real time. This may take 30-90 seconds depending on the Gemini API load.


> **Note for Jupyter / Colab users.** The Jupyter kernel already runs an
> asyncio event loop. CrewAI's `crew.kickoff()` tries to start its own
> loop internally, which raises `RuntimeError: Agent execution was invoked
> synchronously from within a running event loop`. The standard fix is to
> call `nest_asyncio.apply()` **once** at the top of the notebook (or right
> before `kickoff()`). The cell below does that for you.


In [13]:
"""Workaround for running CrewAI inside an existing event loop

    (Jupyter, Google Colab, VS Code interactive window, etc.)."""

try:

    import nest_asyncio

    nest_asyncio.apply()

    print("nest_asyncio applied: CrewAI can now run inside this kernel.")

except ModuleNotFoundError:

    print("nest_asyncio not installed. Run:  pip install nest-asyncio")


nest_asyncio applied: CrewAI can now run inside this kernel.


In [14]:
"""Kick off the swarm. The string passed via `inputs={'topic': ...}` is
    substituted into every '{topic}' placeholder in the task prompts.

Fix: The notebook is running inside an active asyncio event loop (Jupyter/Colab),
so calling crew.kickoff() can raise:
  RuntimeError: Agent execution was invoked synchronously from within a running event loop

We resolve this by using the asynchronous API crew.kickoff_async() and running it
via asyncio.run(...) when possible, or falling back to awaiting it inside the
already-running loop using nest_asyncio-compatible patterns.
"""

from datetime import datetime
import asyncio

# Reuse the existing 'crew' and 'topic' variables defined earlier in the notebook.
# If 'topic' isn't defined yet, provide a sensible default.
try:
    topic  # type: ignore[name-defined]
except NameError:  # define a default topic if not present
    topic = "The benefits of local-first AI agents"

print(f"Starting crew at {datetime.now().isoformat(timespec='seconds')}")
print(f"Topic: {topic!r}\n")

async def _run_crew_async():
    # kickoff_async returns the same result object as kickoff, but non-blocking
    return await crew.kickoff_async(inputs={"topic": topic})

result = None

try:
    # If we're NOT inside an active event loop, use asyncio.run for cleanliness
    loop = asyncio.get_running_loop()
    # If we got here, a loop is already running (typical in Jupyter). Create a task and run it.
    # nest_asyncio.apply() was executed earlier in the notebook; awaiting via ensure_future() works.
    future = asyncio.ensure_future(_run_crew_async())
    result = loop.run_until_complete(future) if hasattr(loop, "run_until_complete") else None
    if result is None:
        # Fallback: directly await via a temporary helper in notebooks
        async def _await_in_notebook():
            return await _run_crew_async()
        result = loop.create_task(_await_in_notebook())
        # Block until done using asyncio.gather (works with nest_asyncio)
        result = loop.run_until_complete(asyncio.gather(result))[0]
except RuntimeError:
    # No running loop: safe to use asyncio.run
    result = asyncio.run(_run_crew_async())
except Exception as e:
    # Last-resort fallback for exotic environments: try nest_asyncio inline await
    try:
        import nest_asyncio  # type: ignore
        nest_asyncio.apply()
        result = asyncio.get_event_loop().run_until_complete(_run_crew_async())
    except Exception as e2:
        raise e2 from e

print(f"\nFinished at {datetime.now().isoformat(timespec='seconds')}")
print(f"Result type: {type(result).__name__}")

# Optionally display the textual content if present
try:
    # CrewAI often returns a string or an object with .raw or .json()
    if isinstance(result, str):
        print("\n--- Final Output ---\n")
        print(result)
    elif hasattr(result, "raw"):
        print("\n--- Final Output (.raw) ---\n")
        print(result.raw)
    else:
        print("\nNote: Result printed above is not a plain string; inspect `result` for details.")
except Exception:
    pass

Starting crew at 2026-06-05T10:20:44
Topic: 'The benefits of local-first AI agents'



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 8d18f4b9-44c8-4459-9c69-8ad3ee8af206                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the topic 'The benefits of local-first AI agents'. Produce a bullet list of 5-7 concrete,       │
│  verifiable points that would make a strong blog post. For each point, include a one-sentence rationale.        │
│  ID: f9214bc0-9839-46ca-a023-8726f8acb546                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Task: Research the topic 'The benefits of local-first AI agents'. Produce a bullet list of 5-7 concrete,       │
│  verifiable points that would make a strong blog post. For each point, include a one-sentence rationale.        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: olostep_web_search                                                                                       │
│  Args: {'scrape': True, 'query': 'benefits of local-first AI agents', 'include_domains': None, 'limit': 5}      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Coordinator                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Blog Post                                                                                                   │
│                                                                                                                 │
│  # The Case for Local-First AI Agents: Privacy, Zero API Fees, and Infinite Loops                               │
│                                                                                                                 │
│  What if your AI agents could run indefinitely, secure your sensitive data, and cost absolutely nothing in API  │
│  fees?                                                                                                          │
│                                                                                                                 │
│  We are rapidly transitioning from simple chat interfaces to complex, autonomous workflows that continuously    │
│  loop, self-correct, and call tools. While cloud-hosted LLMs proved the concept, the architectural tide is      │
│  shifting toward the edge. Local-first AI agents—powered by robust open-weights models and runtimes like        │
│  Ollama or llama.cpp—are emerging as the standard for building fast, secure, and highly cost-effective          │
│  intelligent applications.                                                                                      │
│                                                                                                                 │
│  ### 1. Ironclad Privacy and Compliance                                                                         │
│  Processing sensitive data like proprietary source code, medical records, or financial documents via            │
│  third-party APIs introduces severe compliance risks. Local-first agents mitigate this by processing data       │
│  entirely on your own hardware.                                                                                 │
│                                                                                                                 │
│  Keeping data within a secure local perimeter simplifies compliance with strict regulatory frameworks like      │
│  GDPR, HIPAA, and SOC 2. It also guarantees that your intellectual property is never used to train third-party  │
│  models. By eliminating external network hops, you inherently harden your application's security posture.       │
│                                                                                                                 │
│  ### 2. Predictable Economics (Zero API Bill Shock)                                                             │
│  Agentic workflows are notoriously chatty. Solving a single development or analytical task often requires       │
│  dozens of recursive reasoning steps, vector searches, and self-correction loops. In the cloud, these           │
│  multi-step processes cause token usage—and costs—to scale rapidly, leading to unpredictable billing            │
│  surprises.                                                                                                     │
│                                                                                                                 │
│  Running highly capable open-weights models like Llama 3.1 or Qwen 2.5 locally eliminates this financial        │
│  bottleneck. With zero marginal cost per token, you can

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Look at the full chain (research notes, draft, polished post). Produce a final answer to the user that   │
│  contains: (1) the polished blog post, and (2) a 3-bullet 'Key Takeaways' summary.                              │
│  Agent: Content Coordinator                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Finished at 2026-06-05T10:22:20
Result type: CrewOutput

--- Final Output (.raw) ---

## Blog Post

# The Case for Local-First AI Agents: Privacy, Zero API Fees, and Infinite Loops

What if your AI agents could run indefinitely, secure your sensitive data, and cost absolutely nothing in API fees?

We are rapidly transitioning from simple chat interfaces to complex, autonomous workflows that continuously loop, self-correct, and call tools. While cloud-hosted LLMs proved the concept, the architectural tide is shifting toward the edge. Local-first AI agents—powered by robust open-weights models and runtimes like Ollama or llama.cpp—are emerging as the standard for building fast, secure, and highly cost-effective intelligent applications.

### 1. Ironclad Privacy and Compliance
Processing sensitive data like proprietary source code, medical records, or financial documents via third-party APIs introduces severe compliance risks. Local-first agents mitigate this by processing data entirely 

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 8d18f4b9-44c8-4459-9c69-8ad3ee8af206                                                                       │
│  Final Output: ## Blog Post                                                                                     │
│                                                                                                                 │
│  # The Case for Local-First AI Agents: Privacy, Zero API Fees, and Infinite Loops                               │
│                                                                                                                 │
│  What if your AI agents could run indefinitely, secure your sensitive data, and cost absolutely nothing in API  │
│  fees?                                                                                                          │
│                                                                                                                 │
│  We are rapidly transitioning from simple chat interfaces to complex, autonomous workflows that continuously    │
│  loop, self-correct, and call tools. While cloud-hosted LLMs proved the concept, the architectural tide is      │
│  shifting toward the edge. Local-first AI agents—powered by robust open-weights models and runtimes like        │
│  Ollama or llama.cpp—are emerging as the standard for building fast, secure, and highly cost-effective          │
│  intelligent applications.                                                                                      │
│                                                                                                                 │
│  ### 1. Ironclad Privacy and Compliance                                                                         │
│  Processing sensitive data like proprietary source code, medical records, or financial documents via            │
│  third-party APIs introduces severe compliance risks. Local-first agents mitigate this by processing data       │
│  entirely on your own hardware.                                                                                 │
│                                                                                                                 │
│  Keeping data within a secure local perimeter simplifies compliance with strict regulatory frameworks like      │
│  GDPR, HIPAA, and SOC 2. It also guarantees that your intellectual property is never used to train third-party  │
│  models. By eliminating external network hops, you inherently harden your application's security posture.       │
│                                                                                                                 │
│  ### 2. Predictable Economics (Zero API Bill Shock)                                                             │
│  Agentic workflows are notoriously chatty. Solving a single development or analytical task often requires       │
│  dozens of recursive reasoning steps, vector searches, and self-correction loops. In the cloud, these           │
│  multi-step processes cause token usage—and costs—to scale rapidly, leading to unpredictable billing            │
│  surprises.                                                                                                     │
│                                                                                                                 │
│  Running highly capable open-weights models like Llama 3.1 or Qwen 2.5 locally eliminates this financial        │
│  bottleneck. With zero marginal cost per token, you ca

## 10. Display and explain the final output

CrewAI returns a `CrewOutput` object. The most useful attribute is `.raw`, which is the final answer string produced by the last task in the chain. The intermediate task outputs are also available on each `TaskOutput` if you want to inspect them.

In our chain the **last task** belongs to the `Manager` agent, so `.raw` contains both the polished blog post and the 3-bullet key-takeaways summary we asked for in `coordinate_task.expected_output`.


In [15]:
"""Pretty-print the final result and expose the individual task outputs."""

from IPython.display import Markdown, display



print("=" * 70)

print("FINAL ANSWER (manager agent)")

print("=" * 70)

display(Markdown(result.raw))



print()

print("=" * 70)

print("TASK-BY-TASK TRACE")

print("=" * 70)

for i, task_out in enumerate(result.tasks_output, start=1):

    agent_name = task_out.agent if isinstance(task_out.agent, str) else task_out.agent

    print(f"\n--- Task {i}: {agent_name} ---")

    # Truncate the long middle outputs so the trace stays readable.

    preview = task_out.raw.strip().replace("\n", " ")

    print(preview[:200] + ("..." if len(preview) > 200 else ""))


FINAL ANSWER (manager agent)


## Blog Post

# The Case for Local-First AI Agents: Privacy, Zero API Fees, and Infinite Loops

What if your AI agents could run indefinitely, secure your sensitive data, and cost absolutely nothing in API fees?

We are rapidly transitioning from simple chat interfaces to complex, autonomous workflows that continuously loop, self-correct, and call tools. While cloud-hosted LLMs proved the concept, the architectural tide is shifting toward the edge. Local-first AI agents—powered by robust open-weights models and runtimes like Ollama or llama.cpp—are emerging as the standard for building fast, secure, and highly cost-effective intelligent applications.

### 1. Ironclad Privacy and Compliance
Processing sensitive data like proprietary source code, medical records, or financial documents via third-party APIs introduces severe compliance risks. Local-first agents mitigate this by processing data entirely on your own hardware. 

Keeping data within a secure local perimeter simplifies compliance with strict regulatory frameworks like GDPR, HIPAA, and SOC 2. It also guarantees that your intellectual property is never used to train third-party models. By eliminating external network hops, you inherently harden your application's security posture.

### 2. Predictable Economics (Zero API Bill Shock)
Agentic workflows are notoriously chatty. Solving a single development or analytical task often requires dozens of recursive reasoning steps, vector searches, and self-correction loops. In the cloud, these multi-step processes cause token usage—and costs—to scale rapidly, leading to unpredictable billing surprises. 

Running highly capable open-weights models like Llama 3.1 or Qwen 2.5 locally eliminates this financial bottleneck. With zero marginal cost per token, you can run complex, infinite reasoning loops and iterate on your agentic pipelines all day without monitoring a billing dashboard.

### 3. Ultra-Low Latency and Deep System Integration
Unpredictable network roundtrips and API queue latency degrade the user experience for interactive tasks like real-time code autocomplete. Local-first agents leverage direct hardware acceleration—such as Apple Silicon's Unified Memory or on-device GPUs—to deliver near-instant, sub-second responses. 

Furthermore, local execution allows frameworks like Open Interpreter or local LangGraph runtimes to interact directly with your local file systems and databases without the constraints of cloud sandboxing. This architecture also ensures offline resilience; your critical agents keep working flawlessly even during network outages.

---

By shifting the computational load of agentic AI to local hardware, developers unlock unmatched privacy, predictable economics, and blistering performance. It is time to stop renting cloud intelligence and start building resilient, autonomous systems that you truly own.

## Key Takeaways

* **Ironclad Security & Compliance**: Processing data entirely on local hardware ensures sensitive files, proprietary code, and records remain private, easily meeting strict GDPR, HIPAA, and SOC 2 compliance standards.
* **Elimination of API Cost Scaling**: Moving away from cloud-hosted models allows developers to run complex, multi-step, and recursive agent loops endlessly with zero marginal token costs.
* **Blazing-Fast, Offline-Resilient Performance**: Utilizing local hardware acceleration bypasses network latency and sandbox constraints, delivering sub-second response times and continuous operational reliability even without an internet connection.


TASK-BY-TASK TRACE

--- Task 1: Senior Research Analyst ---
* **Absolute Data Privacy and Regulatory Compliance**: By processing sensitive user data entirely on local hardware using tools like Ollama or llama.cpp, local-first AI agents ensure that proprietary ...

--- Task 2: Blog Post Writer ---
What if your AI agents could run indefinitely, secure your sensitive data, and cost you absolutely nothing in API fees?  As developers, we’re rapidly transitioning from simple chat interfaces to compl...

--- Task 3: Senior Editor ---
# The Case for Local-First AI Agents: Privacy, Zero API Fees, and Infinite Loops  What if your AI agents could run indefinitely, secure your sensitive data, and cost absolutely nothing in API fees?  W...

--- Task 4: Content Coordinator ---
## Blog Post  # The Case for Local-First AI Agents: Privacy, Zero API Fees, and Infinite Loops  What if your AI agents could run indefinitely, secure your sensitive data, and cost absolutely nothing i...


## 11. Error handling and debugging tips

When something goes wrong with a CrewAI + Gemini pipeline, the error usually falls into one of four buckets. Here is a quick troubleshooting table you can come back to.

| Symptom | Likely cause | Fix |
| --- | --- | --- |
| `RuntimeError: No Gemini API key found` | `.env` not loaded, or key still set to the placeholder | Verify `.env` exists in CWD or its parent, and the value is your real key |
| `litellm.AuthenticationError: geminiException` | API key invalid, revoked, or restricted to a different project | Create a new key at [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey) |
| `404 models/gemini-3.5-flash not found` | Google rotated the model name on the API | Try `gemini/gemini-2.5-flash` (the previous stable Flash) or check [ai.google.dev/gemini-api/docs/models](https://ai.google.dev/gemini-api/docs/models) |
| `litellm.BadRequestError: ... models/ ...` | LiteLLM is double-prefixing the model id | Make sure your `LLM(model=...)` string is `"gemini/gemini-3.5-flash"` — only one `gemini/` prefix. See [crewAI#2645](https://github.com/crewAIInc/crewAI/issues/2645) |
| Agents produce empty / very short answers | Temperature too high, or `expected_output` is too vague | Lower `temperature` to `0.3`-`0.5` and tighten the `expected_output` string |
| `ModuleNotFoundError: google.generativeai` | You installed the legacy SDK | `pip install -U google-genai` instead (the legacy package was retired 30 Nov 2025) |
| `RuntimeError: Agent execution was invoked synchronously ... event loop` | Jupyter / Colab already has an asyncio loop running, and CrewAI tried to start a new one | `pip install nest-asyncio` then call `nest_asyncio.apply()` once before `crew.kickoff()` |

**Two debug toggles that save hours:**

```python
import os
os.environ["LITELLM_LOG"] = "DEBUG"     # full request/response logging
import litellm; litellm._turn_on_debug() # alternative, even more verbose
```

Adding either of these before `crew.kickoff()` will print the exact request CrewAI sends to Google, which makes 90% of the remaining issues obvious.


In [16]:
"""Robust wrapper around crew.kickoff() — drop-in replacement.

    Demonstrates the standard error-handling pattern for a real app."""

import litellm



def run_crew_safely(crew, inputs, debug: bool = False):

    """Run a Crew and return its result, with friendly error messages."""

    if debug:

        os.environ["LITELLM_LOG"] = "DEBUG"

        litellm._turn_on_debug()

    try:

        return crew.kickoff(inputs=inputs)

    except litellm.AuthenticationError as e:

        raise RuntimeError(

            "Gemini rejected the API key. Check that GEMINI_API_KEY (or "

            "GOOGLE_API_KEY) in your .env is correct and active."

        ) from e

    except litellm.NotFoundError as e:

        raise RuntimeError(

            f"The Gemini model '{GEMINI_MODEL}' was not found. "

            f"Google may have renamed it; try 'gemini/gemini-2.5-flash'."

        ) from e

    except litellm.RateLimitError as e:

        raise RuntimeError(

            "You hit a Gemini rate limit. Wait a minute and try again, "

            "or reduce the number of tasks / agents in the crew."

        ) from e



print("Helper defined. You can now call `run_crew_safely(crew, {'topic': '...'})`.")


Helper defined. You can now call `run_crew_safely(crew, {'topic': '...'})`.


In [17]:
"""Alternative: run the crew via the async API directly.

    Use this in a notebook that is already async (e.g. inside FastAPI,

    Jupyter with `asyncio.run()`, or any `async def` context)."""

import asyncio



async def run_async():

    return await crew.kickoff_async(inputs={"topic": topic})



try:

    loop = asyncio.get_event_loop()

    if loop.is_running():

        print("Event loop already running — use the nest_asyncio fix above.")

    else:

        result_async = loop.run_until_complete(run_async())

        print("Async run finished. Length:", len(result_async.raw))

except RuntimeError:

    print("No event loop in this thread; safe to call kickoff() synchronously.")


Event loop already running — use the nest_asyncio fix above.


## 12. Optional improvements

The basic swarm works, but there are a handful of small upgrades that make it dramatically more useful in real projects. Pick the ones that match your goal.


### 12.1 Save the final post to a file

Most of the time you want the final output to land on disk so you can publish, archive, or post-process it. This snippet writes the polished post to `output/<slug>.md`.


In [18]:
"""Save the final post to a markdown file."""

import re

from pathlib import Path



output_dir = Path("output")

output_dir.mkdir(exist_ok=True)



# Turn 'The benefits of local-first AI agents' into a filename-safe slug.

slug = re.sub(r"[^a-z0-9]+", "-", topic.lower()).strip("-")

out_path = output_dir / f"{slug}.md"



out_path.write_text(result.raw, encoding="utf-8")

print(f"Saved {len(result.raw):,} characters to {out_path.resolve()}")


Saved 3,555 characters to /datasets/_deepnote_work/output/the-benefits-of-local-first-ai-agents.md


### 12.2 Add a tool (e.g. a web search)

In this notebook the Researcher relies entirely on what Gemini already knows. If you want it to fetch *fresh* information, give it a tool. CrewAI ships a [SerperDevTool](https://docs.crewai.com/tools/serperdevtool) that needs only a `SERPER_API_KEY` in your `.env`.

```python
from crewai_tools import SerperDevTool

researcher.tools = [SerperDevTool()]   # now the researcher can search the web
```

Other handy tools you can mix in the same way: `WebsiteSearchTool`, `FileReadTool`, `ScrapeWebsiteTool`, and `CodeInterpreterTool`. See the [CrewAI tools catalogue](https://docs.crewai.com/en/concepts/tools).


### 12.3 Enable shared memory

Set `memory=True` on the `Crew` to let agents remember previous task outputs and the conversation history. This is what allows the writer to build on the researcher's exact phrasing, and the reviewer to focus on deltas rather than re-reading the entire draft.

```python
crew = Crew(agents=[...], tasks=[...], process=Process.sequential, memory=True)
```

By default, memory is short-term only. To persist memory across runs, set `memory=True` *and* configure a storage backend (Chroma, SQLite, etc.) — see the [memory docs](https://docs.crewai.com/en/concepts/memory).


### 12.4 Switch to a hierarchical (manager-driven) process

Right now we use `Process.sequential` — tasks run in a fixed order. If you want the Manager agent to dynamically decide what should run next (and re-delegate on failure), switch to `Process.hierarchical` and point the crew at a `manager_llm`:

```python
crew = Crew(
    agents=[researcher, writer, reviewer, manager],
    tasks=tasks,
    process=Process.hierarchical,
    manager_llm=gemini_llm,    # <-- required for hierarchical
    verbose=True,
)
```

Hierarchical mode is more flexible but slower and more expensive (every delegation is another LLM call). It shines for open-ended research tasks; for a known 4-step pipeline like ours, sequential is usually the right choice.


### 12.5 Change the model per agent

Different agents benefit from different models. You can pass a different `LLM(...)` object to each agent — e.g. use a faster/cheaper Flash for the writer and reviewer, and a stronger Pro model for the researcher if you have access to one:

```python
from crewai import LLM

fast_llm = LLM(model="gemini/gemini-3.5-flash", temperature=1.0)
strong_llm = LLM(model="gemini/gemini-2.5-pro", temperature=0.4)

researcher.llm = strong_llm   # deep research
writer.llm     = fast_llm     # fast drafting
reviewer.llm   = fast_llm     # fast editing
manager.llm    = fast_llm     # fast coordination
```


## 13. Conclusion & next steps

You have just built a complete **4-agent CrewAI swarm** powered by **Gemini 3.5 Flash**:

1. A *Researcher* that pulls together the key points on a topic.
2. A *Writer* that turns those points into a blog post.
3. A *Reviewer* that polishes the draft.
4. A *Manager* that delivers the final, formatted answer to you.

From here, the natural next steps are:

- **Add real tools** (web search, file reading, code execution) so the researcher can pull in *live* data instead of relying on the LLM's training cutoff.
- **Persist memory** with Chroma or SQLite so each run can build on the previous one.
- **Tighten the prompts** — the quality of a CrewAI swarm is dominated by the quality of the `role`, `goal`, `backstory`, and `expected_output` strings you write.
- **Swap the LLM** — change one line (`GEMINI_MODEL`) to point the whole crew at `gemini-2.5-pro`, an OpenAI model, a local Ollama model, etc.
- **Wrap it in a Flow** — for production, use [CrewAI Flows](https://docs.crewai.com/en/concepts/flows) to add state, retries, and human-in-the-loop checkpoints.

Happy swarming!


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=35be27b9-5552-4766-b3f4-78405029aa1d' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>